# EDA

In [5]:
import pandas as pd

# Load the preprocessed order-level dataset
df = pd.read_csv(
    "../01_data/DataCo_order_level.csv",
    parse_dates=[
        "order date (DateOrders)",
        "shipping date (DateOrders)",
    ],
)

# Confirm the dimensions of the order-level dataset
df.shape

(65752, 140)

In [10]:
# Verify that the dataset contains one record per order
assert df["Order Id"].is_unique

In [6]:
# Examine the distribution of order statuses
order_status_counts = df["Order Status"].value_counts()

order_status_counts

Order Status
COMPLETE           21716
PENDING_PAYMENT    14382
PROCESSING          7901
PENDING             7321
CLOSED              7249
ON_HOLD             3624
SUSPECTED_FRAUD     1488
CANCELED            1367
PAYMENT_REVIEW       704
Name: count, dtype: int64

In [7]:
# Compare order status with recorded delivery status
pd.crosstab(
    df["Order Status"],
    df["Delivery Status"],
)

Delivery Status,Advance shipping,Late delivery,Shipping canceled,Shipping on time
Order Status,,,,
CANCELED,0,0,1367,0
CLOSED,1777,4112,0,1360
COMPLETE,5174,12521,0,4021
ON_HOLD,858,2045,0,721
PAYMENT_REVIEW,159,405,0,140
PENDING,1797,4215,0,1309
PENDING_PAYMENT,3421,8256,0,2705
PROCESSING,1941,4494,0,1466
SUSPECTED_FRAUD,0,0,1488,0


In [8]:
# Compare late-delivery rates across order statuses
pd.crosstab(
    df["Order Status"],
    df["Late_delivery_risk"],
    normalize="index",
).round(3)

Late_delivery_risk,0,1
Order Status,,
CANCELED,1.000,0.000
CLOSED,0.433,0.567
COMPLETE,0.423,0.577
ON_HOLD,0.436,0.564
PAYMENT_REVIEW,0.425,0.575
PENDING,0.424,0.576
PENDING_PAYMENT,0.426,0.574
PROCESSING,0.431,0.569
SUSPECTED_FRAUD,1.000,0.000


In [9]:
# Summarize the temporal coverage of each order status
status_dates = (
    df.groupby("Order Status")["order date (DateOrders)"]
    .agg(["count", "min", "max"])
    .sort_values("max", ascending=False)
)

status_dates

,count,min,max
Order Status,,,
CLOSED,7249,2015-01-01 00:00:00,2018-01-31 23:38:00
PENDING_PAYMENT,14382,2015-01-01 00:21:00,2018-01-31 23:17:00
COMPLETE,21716,2015-01-01 01:24:00,2018-01-31 22:56:00
PROCESSING,7901,2015-01-01 02:27:00,2018-01-31 22:14:00
PENDING,7321,2015-01-01 07:00:00,2018-01-31 19:47:00
SUSPECTED_FRAUD,1488,2015-01-01 23:49:00,2018-01-31 16:38:00
ON_HOLD,3624,2015-01-01 15:45:00,2018-01-30 22:04:00
PAYMENT_REVIEW,704,2015-01-01 03:30:00,2018-01-30 11:12:00
CANCELED,1367,2015-01-01 17:10:00,2018-01-30 10:09:00


**Order Status note:** Nonterminal status labels such as PENDING_PAYMENT, PROCESSING, and ON_HOLD occur throughout the full 2015–2018 period despite having realized shipping outcomes. Therefore, Order Status cannot be interpreted as a snapshot of unresolved orders at dataset extraction. The field is retained for exploratory analysis, but its temporal meaning and suitability as a predictive feature require further evaluation.